# Volatility forecast comparison: training and validation (Phase 14, step 2)

Compares four 5-day volatility forecasts for BTC/USDT and SOL/USDT on the **validation period only**. Scaled `vol_20`, the current dashboard forecast, is the one to beat. The test period (2024 onward) is not used: daily and 15-minute data are cut at 2023-12-31 before any target or forecast is computed.

**Targets at day t** (daily units)
- **Primary:** realized volatility over the next 5 days, √(mean of RV_d² over days t+1 to t+5), where RV_d comes from `daily_realized_vol` (15-minute data). NaN if any of those days is missing.
- **Secondary (Phase 4 target):** √(mean of squared daily close-to-close log returns over days t+1 to t+5).

**Forecasts at day t** (each uses only information known at the close of day t)
1. **Raw `vol_20`.**
2. **Scaled `vol_20`:** `scaled_vol_forecast(df, 5)`, exactly as the dashboard uses it.
3. **Scaled realized volatility:** √(mean of RV_d² over the last 20 days), times a scale factor learned point in time against the primary target. The factor uses the same method as `scaled_vol_forecast`: the expanding mean of log(target / RV20), shifted by 5 days, so only completed outcomes are used.
4. **HAR:** a linear regression of log(primary target) on log RV_t, log(mean RV over the last 5 days) and log(mean RV over the last 22 days), fitted on the training period only. Forecast = exp(prediction).
   - The 5- and 22-day windows include day t.
   - A window with any missing RV_d gives NaN.

**Periods** (day t)

| Asset | Training | Validation |
|---|---|---|
| BTC | t ≤ 2021-12-26 | 2022-01-01 ≤ t ≤ 2023-12-26 |
| SOL | t ≤ 2022-12-26 | 2023-01-01 ≤ t ≤ 2023-12-26 |

- **Purge:** the last 5 days before each boundary are dropped, so every training target ends by the training end and every validation target ends by 2023-12-31.
- **Fair comparison:** metrics use the same days for every forecast: validation days where the target and all four forecasts exist.
- **Error measures:** MAE and RMSE in percentage points of daily volatility. Improvement = (error of scaled `vol_20` − error of the forecast) / error of scaled `vol_20` × 100.

## Step 0: Imports and settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.volatility import daily_realized_vol, scaled_vol_forecast

DATA_DIR = Path.cwd().parent / "data"
HORIZON = 5
DATA_END = pd.Timestamp("2023-12-31", tz="UTC")          # nothing from 2024 onward is loaded
PURGE = pd.Timedelta(days=HORIZON)
PERIODS = {
    "BTC/USDT": {"train_end": "2021-12-31", "val_start": "2022-01-01", "val_end": "2023-12-31"},
    "SOL/USDT": {"train_end": "2022-12-31", "val_start": "2023-01-01", "val_end": "2023-12-31"},
}
FORECASTS = ["raw vol_20", "scaled vol_20", "scaled RV", "HAR"]
TARGETS = {"primary": "target_rv", "secondary": "target_cc"}
HAR_FEATURES = ["log_rv_1", "log_rv_5", "log_rv_22"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)

## Step 1: Data, targets and point-in-time forecasts 1–3

In [2]:
def build(symbol):
    """Daily frame with targets, HAR inputs and forecasts 1-3, from data cut at DATA_END."""
    daily = add_features(load_processed(symbol, DATA_DIR)).loc[:DATA_END]
    m15 = load_processed(symbol, DATA_DIR, timeframe="15m")
    m15 = m15[m15.index < DATA_END + pd.Timedelta(days=1)]
    rv = daily_realized_vol(m15)
    rv = rv.set_axis(rv.index.as_unit(daily.index.unit)).reindex(daily.index)

    f = pd.DataFrame(index=daily.index)
    f["rv"] = rv
    # Targets: next 5 days (NaN if any day is missing or beyond the cut).
    f["target_rv"] = np.sqrt((rv ** 2).rolling(HORIZON).mean()).shift(-HORIZON)
    f["target_cc"] = np.sqrt((daily["log_return"] ** 2).rolling(HORIZON).mean()).shift(-HORIZON)
    # Forecasts 1 and 2.
    f["raw vol_20"] = daily["vol_20"]
    f["scaled vol_20"] = scaled_vol_forecast(daily, HORIZON)
    # Forecast 3: RV over the last 20 days, scaled point in time against the primary target.
    rv20 = np.sqrt((rv ** 2).rolling(20).mean())
    scale = np.exp(np.log(f["target_rv"] / rv20).expanding().mean().shift(HORIZON))
    f["scaled RV"] = rv20 * scale
    # HAR inputs (windows include day t).
    f["log_rv_1"] = np.log(rv)
    f["log_rv_5"] = np.log(rv.rolling(5).mean())
    f["log_rv_22"] = np.log(rv.rolling(22).mean())
    return f


frames = {symbol: build(symbol) for symbol in PERIODS}
for symbol, f in frames.items():
    print(f"{symbol}: {f.index[0].date()} -> {f.index[-1].date()} ({len(f):,} days); "
          f"RV_d missing on {int(f['rv'].isna().sum())} days; last primary target on "
          f"{f['target_rv'].last_valid_index().date()}")

BTC/USDT: 2017-08-17 -> 2023-12-31 (2,328 days); RV_d missing on 27 days; last primary target on 2023-12-26
SOL/USDT: 2020-08-11 -> 2023-12-31 (1,238 days); RV_d missing on 9 days; last primary target on 2023-12-26


## Step 2: HAR fitted on the training period only

In [3]:
har_models, coef_rows = {}, []
for symbol, f in frames.items():
    p = PERIODS[symbol]
    train_end = pd.Timestamp(p["train_end"], tz="UTC") - PURGE
    train = f.loc[:train_end].dropna(subset=HAR_FEATURES + ["target_rv"])
    model = LinearRegression().fit(train[HAR_FEATURES], np.log(train["target_rv"]))
    har_models[symbol] = model

    usable = f[HAR_FEATURES].notna().all(axis=1)
    f["HAR"] = np.nan
    f.loc[usable, "HAR"] = np.exp(model.predict(f.loc[usable, HAR_FEATURES]))
    coef_rows.append({"asset": symbol, "training days": len(train),
                      "first": train.index[0].date(), "last": train.index[-1].date(),
                      "intercept": model.intercept_,
                      **dict(zip(["b_log_rv_1", "b_log_rv_5", "b_log_rv_22"], model.coef_))})

har_coefficients = pd.DataFrame(coef_rows).set_index("asset")
print(har_coefficients.to_string(float_format="{:.4f}".format))

          training days       first        last  intercept  b_log_rv_1  b_log_rv_5  b_log_rv_22
asset                                                                                          
BTC/USDT           1030  2017-09-28  2021-12-26    -0.7103      0.2404      0.2177       0.3253
SOL/USDT            683  2020-09-02  2022-12-26    -0.9000      0.3183      0.3012       0.0480


## Step 3: Validation errors against both targets

Common sample per asset and target: validation days where the target and all four forecasts exist.

In [4]:
def period_rows(f, start, end):
    return f.loc[pd.Timestamp(start, tz="UTC"):pd.Timestamp(end, tz="UTC") - PURGE]


def errors(sample, target_col):
    rows = {}
    for name in FORECASTS:
        e = sample[name] - sample[target_col]
        rows[name] = {"MAE_pp": 100 * e.abs().mean(), "RMSE_pp": 100 * np.sqrt((e ** 2).mean())}
    table = pd.DataFrame(rows).T
    base = table.loc["scaled vol_20"]
    table["MAE_improvement_%"] = 100 * (base["MAE_pp"] - table["MAE_pp"]) / base["MAE_pp"]
    table["RMSE_improvement_%"] = 100 * (base["RMSE_pp"] - table["RMSE_pp"]) / base["RMSE_pp"]
    return table


val_tables, val_samples = {}, {}
for symbol, f in frames.items():
    p = PERIODS[symbol]
    val = period_rows(f, p["val_start"], p["val_end"])
    for target, col in TARGETS.items():
        sample = val.dropna(subset=FORECASTS + [col])
        val_samples[(symbol, target)] = sample
        val_tables[(symbol, target)] = errors(sample, col)
        print(f"\n{symbol}, {target} target: {len(sample)} validation days "
              f"({sample.index[0].date()} -> {sample.index[-1].date()})")
        print(val_tables[(symbol, target)].to_string(float_format="{:.3f}".format))


BTC/USDT, primary target: 698 validation days (2022-01-01 -> 2023-12-26)
               MAE_pp  RMSE_pp  MAE_improvement_%  RMSE_improvement_%
raw vol_20      0.836    1.222             -7.179              -1.075
scaled vol_20   0.780    1.209              0.000               0.000
scaled RV       0.770    1.187              1.357               1.789
HAR             0.691    1.018             11.398              15.759

BTC/USDT, secondary target: 703 validation days (2022-01-01 -> 2023-12-26)
               MAE_pp  RMSE_pp  MAE_improvement_%  RMSE_improvement_%
raw vol_20      1.309    1.722            -10.597              -4.019
scaled vol_20   1.183    1.656              0.000               0.000
scaled RV       1.205    1.632             -1.862               1.431
HAR             1.167    1.529              1.386               7.650

SOL/USDT, primary target: 333 validation days (2023-01-01 -> 2023-12-26)
               MAE_pp  RMSE_pp  MAE_improvement_%  RMSE_improvement_%
raw vo

## Step 4: Validation MAE by year

In [5]:
rows = []
for (symbol, target), sample in val_samples.items():
    col = TARGETS[target]
    for year, g in sample.groupby(sample.index.year):
        mae = {name: 100 * (g[name] - g[col]).abs().mean() for name in FORECASTS}
        rows.append({"asset": symbol, "target": target, "year": year, "days": len(g),
                     **{f"MAE_pp {name}": v for name, v in mae.items()},
                     "HAR improvement_%": 100 * (mae["scaled vol_20"] - mae["HAR"]) / mae["scaled vol_20"],
                     "scaled RV improvement_%": 100 * (mae["scaled vol_20"] - mae["scaled RV"]) / mae["scaled vol_20"]})
by_year = pd.DataFrame(rows).set_index(["asset", "target", "year"])
print(by_year.to_string(float_format="{:.3f}".format))

                         days  MAE_pp raw vol_20  MAE_pp scaled vol_20  MAE_pp scaled RV  MAE_pp HAR  HAR improvement_%  scaled RV improvement_%
asset    target    year                                                                                                                         
BTC/USDT primary   2022   365              1.026                 0.944             0.906       0.773             18.161                    4.028
                   2023   333              0.629                 0.601             0.620       0.602             -0.256                   -3.247
         secondary 2022   365              1.575                 1.407             1.429       1.334              5.199                   -1.509
                   2023   338              1.022                 0.942             0.964       0.986             -4.770                   -2.432
SOL/USDT primary   2023   333              1.527                 1.555             1.433       1.335             14.169           

## Step 5: HAR overfitting check — training versus validation MAE (primary target)

Training MAE uses the HAR training days; validation MAE uses all validation days where HAR and the primary target exist (not restricted to the common sample).

In [6]:
rows = []
for symbol, f in frames.items():
    p = PERIODS[symbol]
    train = f.loc[:pd.Timestamp(p["train_end"], tz="UTC") - PURGE].dropna(subset=["HAR", "target_rv"])
    val = period_rows(f, p["val_start"], p["val_end"]).dropna(subset=["HAR", "target_rv"])
    rows.append({"asset": symbol,
                 "train days": len(train), "train MAE_pp": 100 * (train["HAR"] - train["target_rv"]).abs().mean(),
                 "validation days": len(val), "validation MAE_pp": 100 * (val["HAR"] - val["target_rv"]).abs().mean()})
har_fit = pd.DataFrame(rows).set_index("asset")
har_fit["validation / train"] = har_fit["validation MAE_pp"] / har_fit["train MAE_pp"]
print(har_fit.to_string(float_format="{:.3f}".format))

          train days  train MAE_pp  validation days  validation MAE_pp  validation / train
asset                                                                                     
BTC/USDT        1030         0.991              698              0.691               0.698
SOL/USDT         683         1.630              333              1.335               0.819


## Validation results

Errors in percentage points of daily volatility. Improvement = % reduction in error relative to scaled vol_20 (positive = better than scaled vol_20). Common sample per asset and target.

### BTC/USDT (validation 2022-01-01 → 2023-12-26)

| Forecast | Primary MAE | Primary RMSE | MAE impr. % | RMSE impr. % | Secondary MAE | Secondary RMSE | MAE impr. % | RMSE impr. % |
|---|---|---|---|---|---|---|---|---|
| raw vol_20 | 0.836 | 1.222 | −7.18 | −1.08 | 1.309 | 1.722 | −10.60 | −4.02 |
| scaled vol_20 | 0.780 | 1.209 | 0.00 | 0.00 | 1.183 | 1.656 | 0.00 | 0.00 |
| scaled RV | 0.770 | 1.187 | 1.36 | 1.79 | 1.205 | 1.632 | −1.86 | 1.43 |
| HAR | 0.691 | 1.018 | 11.40 | 15.76 | 1.167 | 1.529 | 1.39 | 7.65 |

Validation days: primary 698, secondary 703.

### SOL/USDT (validation 2023-01-01 → 2023-12-26)

| Forecast | Primary MAE | Primary RMSE | MAE impr. % | RMSE impr. % | Secondary MAE | Secondary RMSE | MAE impr. % | RMSE impr. % |
|---|---|---|---|---|---|---|---|---|
| raw vol_20 | 1.527 | 2.099 | 1.81 | 3.36 | 1.775 | 2.356 | −3.39 | −0.92 |
| scaled vol_20 | 1.555 | 2.172 | 0.00 | 0.00 | 1.717 | 2.335 | 0.00 | 0.00 |
| scaled RV | 1.433 | 1.932 | 7.89 | 11.07 | 1.696 | 2.142 | 1.22 | 8.28 |
| HAR | 1.335 | 1.703 | 14.17 | 21.60 | 1.816 | 2.234 | −5.78 | 4.31 |

Validation days: primary 333, secondary 338.

### MAE by validation year (improvement over scaled vol_20, %)

| Asset | Target | Year | Days | Scaled vol_20 MAE | Scaled RV MAE | HAR MAE | Scaled RV impr. | HAR impr. |
|---|---|---|---|---|---|---|---|---|
| BTC | primary | 2022 | 365 | 0.944 | 0.906 | 0.773 | 4.03 | 18.16 |
| BTC | primary | 2023 | 333 | 0.601 | 0.620 | 0.602 | −3.25 | −0.26 |
| BTC | secondary | 2022 | 365 | 1.407 | 1.429 | 1.334 | −1.51 | 5.20 |
| BTC | secondary | 2023 | 338 | 0.942 | 0.964 | 0.986 | −2.43 | −4.77 |
| SOL | primary | 2023 | 333 | 1.555 | 1.433 | 1.335 | 7.89 | 14.17 |
| SOL | secondary | 2023 | 338 | 1.717 | 1.696 | 1.816 | 1.22 | −5.78 |

Raw vol_20 MAE by year: BTC primary 1.026 (2022), 0.629 (2023); BTC secondary 1.575, 1.022; SOL primary 1.527; SOL secondary 1.775.

### HAR coefficients (training period only; target log of the primary target)

| Asset | Training days | First → last | Intercept | log RV (1 day) | log mean RV (5 days) | log mean RV (22 days) |
|---|---|---|---|---|---|---|
| BTC | 1,030 | 2017-09-28 → 2021-12-26 | −0.7103 | 0.2404 | 0.2177 | 0.3253 |
| SOL | 683 | 2020-09-02 → 2022-12-26 | −0.9000 | 0.3183 | 0.3012 | 0.0480 |

### HAR training versus validation MAE (primary target)

| Asset | Train days | Train MAE | Validation days | Validation MAE | Validation / train |
|---|---|---|---|---|---|
| BTC | 1,030 | 0.991 | 698 | 0.691 | 0.698 |
| SOL | 683 | 1.630 | 333 | 1.335 | 0.819 |

## Decision (validation gate)

**Decision:** HAR meets the adoption rule on validation: on the primary target it beat scaled vol_20 by 11.4% (BTC) and 14.2% (SOL) in MAE, and by 15.8% and 21.6% in RMSE, without overfitting (validation/train MAE 0.70 and 0.82). Caveats: on the secondary target its MAE gain was +1.4% (BTC) and −5.8% (SOL); scaled vol_20's scale was learned against the secondary target, which partly disadvantages it on the primary target; BTC's gain came from 2022 (2023: −0.3%); SOL's validation covers one year. Scaled realized volatility is not taken forward.

**Held-out test, run once:** HAR is refitted on training plus validation data (purged before 2024-01-01) and compared with raw vol_20 and scaled vol_20 on 2024 onward, on both targets, for BTC and SOL. **Adoption criterion, fixed now:** HAR replaces the dashboard forecast only if its primary-target MAE improvement over scaled vol_20 is positive for both BTC and SOL in the test period; otherwise the current forecast is kept.